# CSS 551 · HW7 · A gradient step, a fit, and a digit's DCT (Python track)

Fill in every function marked `TODO`, run all cells, and submit this notebook **run**, with
the check cell's output visible. The functions and the numbers are the same as the
JavaScript track's (see the homework page); the last cell compares yours with them.

Rules: NumPy is allowed for arrays, dot products and matrix products. No `scipy`, no
autodiff (`torch`, `jax`), no `np.fft` or DCT routines: the point is to write the gradient
and the transform yourself. Keep `data.json` and `expected.json` next to this notebook.

In [ ]:
import json, math
import numpy as np

DATA = json.load(open("data.json"))
EXP = json.load(open("expected.json"))
INPUTS = EXP["inputs"]

In [ ]:
def forward(m, x):
    """m = {"w1", "b1", "w2" (arrays of H), "b2"}; returns (a, f) with
    a[h] = tanh(w1[h] x + b1[h]) and f = b2 + sum_h w2[h] a[h]."""
    # TODO
    return np.zeros(len(m["w1"])), 0.0

In [ ]:
def gradients(m, xs, ys):
    """Backpropagation for the MEAN squared error over the samples. Returns a dict
    {"w1", "b1", "w2" (arrays), "b2", "loss"}."""
    # TODO: per sample, dL/df = 2 (f - y) / n; push it back through w2 and tanh' = 1 - a^2.
    H = len(m["w1"])
    return {"w1": np.zeros(H), "b1": np.zeros(H), "w2": np.zeros(H), "b2": 0.0, "loss": 0.0}

In [ ]:
def sgd_step(m, g, lr):
    """A NEW model with every parameter moved by -lr times its gradient."""
    # TODO
    return {k: np.array(m[k], float) for k in ("w1", "b1", "w2")} | {"b2": m["b2"]}

In [ ]:
def dct2(img, n):
    """Orthonormal 2D DCT-II of an n x n image given row-major (img[y * n + x]).
    Returns C, row-major, C[u * n + v] for vertical frequency u, horizontal v:
    C[u][v] = c(u) c(v) sum_y sum_x img[y][x] cos(pi (x + 0.5) v / n) cos(pi (y + 0.5) u / n),
    c(0) = sqrt(1/n), c(k > 0) = sqrt(2/n). (Loops or a matrix product; not scipy.)"""
    # TODO
    return np.zeros(n * n)

In [ ]:
def idct2_truncated(C, n, k):
    """The image rebuilt from the low-frequency k x k block of C only (u < k and v < k), row-major."""
    # TODO
    return np.zeros(n * n)

In [ ]:
def psnr(a, b):
    """PSNR in dB of b against a, pixel values in [0, 1]: 10 log10(1 / MSE)."""
    # TODO
    return 0.0

## Checks

Run this last. It prints each check with your value and the expected one, and the score on the published inputs. Grading also runs your functions on inputs that are not published.

In [ ]:
def flat(v):
    if isinstance(v, dict):
        raise TypeError("a check returned a dict; return the values in the order the check asks for")
    if isinstance(v, (list, tuple, np.ndarray)):
        out = []
        for x in v:
            out.extend(flat(x))
        return out
    if isinstance(v, (bool, np.bool_)):
        return [bool(v)]
    return [float(v)]

def same(got, want, tol):
    if want is None or got is None:
        return got is None and want is None
    g, w = flat(got), flat(want)
    if len(g) != len(w):
        return False
    return all(abs(a - b) <= tol and not math.isnan(a) for a, b in zip(g, w))

def report(runs):
    score, lines = 0, []
    for c in EXP["checks"]:
        try:
            got, err = runs[c["id"]](), None
        except Exception as e:  # a stub or a bug: the check fails, the rest still run
            got, err = None, repr(e)
        ok = err is None and same(got, c["expected"], c["tol"])
        score += c["points"] if ok else 0
        shown = err if err else np.round(np.array(flat(got), dtype=float), 4).tolist() if got is not None else None
        lines.append(f'{"PASS" if ok else "FAIL"}  {c["id"]:<8} {c["points"]:>3} pts  {c["label"]}\n      yours {shown}\n      expected {c["expected"]}')
    print("\n".join(lines))
    print(f"\nscore on the published inputs: {score} / 100")
    if score == 100:
        print("ALL CHECKS PASS")


T = INPUTS["tiny"]
digit, n = INPUTS.get("digit", DATA["digit"]), INPUTS["dct"]["n"]

def fit_loss():
    m = {k: np.array(DATA["init"][k], float) for k in ("w1", "b1", "w2")} | {"b2": DATA["init"]["b2"]}
    xs, ys = DATA["samples"]["xs"], DATA["samples"]["ys"]
    for _ in range(INPUTS["fit"]["steps"]):
        m = sgd_step(m, gradients(m, xs, ys), INPUTS["fit"]["lr"])
    return gradients(m, xs, ys)["loss"]

def grad_row():
    g = gradients(T, [T["x"]], [T["y"]])
    return [g["loss"], g["w1"], g["b1"], g["w2"], g["b2"]]

report({
    "fwd": lambda: list(forward(T, T["x"])),
    "grad": grad_row,
    "after": lambda: forward(sgd_step(T, gradients(T, [T["x"]], [T["y"]]), T["lr"]), T["x"])[1],
    "fit": fit_loss,
    "dc": lambda: (lambda C: [C[0], C[1], C[n]])(dct2(digit, n)),
    "psnr5": lambda: psnr(digit, idct2_truncated(dct2(digit, n), n, 5)),
    "psnr8": lambda: psnr(digit, idct2_truncated(dct2(digit, n), n, 8)),
})